In [1]:
import torch.nn as nn
import torch


class VGG(nn.Module):
    def __init__(self, num_classes=40, init_weights=False):
        super(VGG, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=3, padding=1),
            nn.ReLU(True),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(True),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(True),
            nn.Conv1d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(True),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Conv1d(256, 512, kernel_size=3, padding=1),
            nn.ReLU(True),
            nn.Conv1d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(True),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Conv1d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(True),
            nn.Conv1d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(True),
            nn.MaxPool1d(kernel_size=2, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(23552, 4096),
            nn.ReLU(True),
            nn.Dropout(p=0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(p=0.5),
            nn.Linear(4096, num_classes)
        )
        if init_weights:
            self._initialize_weights()

    def forward(self, x):
        x = self.conv(x)
        x = torch.flatten(x, start_dim=1)
        x = self.classifier(x)
        return x

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.constant_(m.bias, 0)

In [2]:
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
import numpy as np

def plot_matrix(conf_matrix, dev_list, save_path):
    plt.figure(figsize=(20, 16), dpi=300)
    plt.imshow(conf_matrix, cmap=plt.cm.Blues, interpolation='nearest')
    plt.title('Confusion Matrix')
    plt.colorbar()

    thresh = conf_matrix.max() / 2.
    for i in range(conf_matrix.shape[0]):
        for j in range(conf_matrix.shape[1]):
            plt.text(j, i, conf_matrix[i, j],
                     ha="center", va="center",
                     color="white" if conf_matrix[i, j] > thresh else "black", fontsize=8)

    tick_marks = np.arange(len(dev_list))
    plt.xticks(tick_marks, dev_list, fontsize=8)
    plt.yticks(tick_marks, dev_list, fontsize=8)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')

    plt.gca().set_aspect(aspect='equal')

    plt.savefig(save_path)
    plt.show()

In [ ]:
import os
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import DataLoader
import numpy as np
from sklearn.metrics import f1_score
from sklearn.metrics import recall_score
from sklearn.metrics import confusion_matrix

class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.categories = sorted(os.listdir(data_dir))
        self.data = []
        self.transform = transform
        for category in self.categories:
            category_dir = os.path.join(data_dir, category)
            category_data = sorted(os.listdir(category_dir))
            self.data.extend([(os.path.join(category_dir, file), self.categories.index(category)) for file in category_data])

    def __getitem__(self, index):
        file_path, label = self.data[index]
        data = np.load(file_path)
        image = Image.fromarray(data.astype(np.uint8))
        image = transform(image)
        return image, label

    def __len__(self):
        return len(self.data)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("using {} device.".format(device))

transform = transforms.Compose([ transforms.Grayscale(num_output_channels = 1),
                                     transforms.ToTensor(),
                                     transforms.Normalize((0.5), (0.5))])

train_dataset = CustomDataset("features/train_npy",transform=transform)
train_num = len(train_dataset)
dev_list = train_dataset.categories

batch_size = 32
train_loader = torch.utils.data.DataLoader(train_dataset,
                                               batch_size=batch_size, shuffle=True,
                                               num_workers=0)

validate_dataset = CustomDataset("features/val_npy",transform=transform)
val_num = len(validate_dataset)
validate_loader = torch.utils.data.DataLoader(validate_dataset,
                                                  batch_size=batch_size, shuffle=False,
                                                  num_workers=0)

print("using {} images for training, {} images for validation.".format(train_num, val_num))

net = VGG(num_classes=31,init_weights=True)
net.to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)

epochs = 100
save_path = './VggNet_parameters.pth'
best_f1 = 0.0
train_accurate_list = []
val_accurate_list = []
f1_list = []
recall_list = []

for epoch in range(epochs):
    net.train()
    running_loss = 0.0
    train_acc = 0.0
    for step, data in enumerate(train_loader, start=0):
        images, labels = data
        images = images.reshape(images.shape[0], 1, 1503)
        optimizer.zero_grad()
        outputs = net(images.to(device))
        predict_y = torch.max(outputs, dim=1)[1]
        train_acc += torch.eq(predict_y, labels.to(device)).sum().item()
        loss = loss_function(outputs,labels.to(device))
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        rate = (step + 1) / len(train_loader)
        a = "*" * int(rate * 50)
        b = "." * int((1 - rate) * 50)
        print("\rtrain loss:{:^3.0f}%[{}->{}]{:.3f}".format(int(rate * 100), a, b, loss), end="")
    print()
    train_accurate = train_acc / train_num
    train_accurate_list.append(train_accurate)
    net.eval()
    acc = 0.0  
    val = torch.tensor([])
    pre = torch.tensor([])
    with torch.no_grad():
        for val_data in validate_loader:
            val_images, val_labels = val_data
            val_images = val_images.reshape(val_images.shape[0], 1, 1503)
            outputs = net(val_images.to(device))
            predict_y = torch.max(outputs, dim=1)[1]
            pre = torch.cat([pre.to(device), predict_y.to(device)])
            val = torch.cat([val.to(device), val_labels.to(device)])
            acc += torch.eq(predict_y, val_labels.to(device)).sum().item()
    val_accurate = acc / val_num
    val_accurate_list.append(val_accurate)
    f1 = f1_score(val.cpu(), pre.cpu(), average='macro')
    recall = recall_score(val.cpu(), pre.cpu(), average='macro')

    f1_list.append(f1)
    recall_list.append(recall)
    if f1 > best_f1:
        best_f1 = f1
        best_pre = pre
        best_val = val
        torch.save(net.state_dict(), save_path)
        torch.save(best_pre, 'pre_val_label/best_pre_VggNet.pt')
        torch.save(best_val, 'pre_val_label/best_val_VggNet.pt')
    print('[epoch %d] train_loss: %.3f train_accuracy: %.3f val_accuracy: %.3f  recall: %.3f  f1: %.3f' %
              (epoch + 1, running_loss / step, train_accurate, val_accurate, recall, f1))
    with open("VggNet_result_npy.txt", 'a') as file:
        file.write("[epoch " + str(epoch + 1) + "]" + "  " + "train_accuracy:" + str(train_accurate) + "  " + "val_accuracy:" + str(val_accurate) + "  " + "recall:" + str(recall) + "  " + "f1:" + str(f1) + '\n')
print('Finished Training')
iterations = range(1, len(train_accurate_list) + 1)
with open("VggNet_npy_plt_data.txt", 'a') as file:
    file.write("iterations:" + str(iterations) +
               "train_accurate_list:" + str(train_accurate_list) +
               "val_accurate_list:" + str(val_accurate_list) +
               "f1_list:" + str(f1_list) +
               "recall_list:" + str(recall_list) +
               "dev_list:" + str(dev_list) + '\n')
conf_matrix = confusion_matrix(best_val.cpu(),best_pre.cpu())
plot_matrix(conf_matrix,dev_list,"VggNet_confusion_matrix_npy.png")

using cuda:0 device.
using 82841 images for training, 20695 images for validation.
train loss:100%[**************************************************->]0.614
[epoch 1] train_loss: 1.236 train_accuracy: 0.564 val_accuracy: 0.692  recall: 0.566  f1: 0.533
train loss:100%[**************************************************->]0.869
[epoch 2] train_loss: 0.701 train_accuracy: 0.716 val_accuracy: 0.743  recall: 0.627  f1: 0.617
train loss:100%[**************************************************->]0.872
[epoch 3] train_loss: 0.611 train_accuracy: 0.750 val_accuracy: 0.760  recall: 0.638  f1: 0.635
train loss:100%[**************************************************->]0.399
[epoch 4] train_loss: 0.560 train_accuracy: 0.771 val_accuracy: 0.770  recall: 0.674  f1: 0.667
train loss:100%[**************************************************->]0.591
[epoch 5] train_loss: 0.513 train_accuracy: 0.792 val_accuracy: 0.812  recall: 0.706  f1: 0.709
train loss:100%[**********************************************

train loss:100%[**************************************************->]0.3514
[epoch 48] train_loss: 0.581 train_accuracy: 0.866 val_accuracy: 0.870  recall: 0.736  f1: 0.724
train loss:100%[**************************************************->]0.2215
[epoch 49] train_loss: 0.490 train_accuracy: 0.866 val_accuracy: 0.875  recall: 0.783  f1: 0.775
train loss:100%[**************************************************->]0.283
[epoch 50] train_loss: 0.296 train_accuracy: 0.896 val_accuracy: 0.902  recall: 0.793  f1: 0.795
train loss:100%[**************************************************->]0.262
[epoch 51] train_loss: 0.380 train_accuracy: 0.874 val_accuracy: 0.900  recall: 0.789  f1: 0.802
train loss:100%[**************************************************->]0.239
[epoch 52] train_loss: 0.486 train_accuracy: 0.846 val_accuracy: 0.884  recall: 0.775  f1: 0.787
train loss:100%[**************************************************->]0.149
[epoch 53] train_loss: 0.346 train_accuracy: 0.882 val_accuracy

train loss:100%[**************************************************->]1.400
[epoch 96] train_loss: 0.517 train_accuracy: 0.832 val_accuracy: 0.492  recall: 0.333  f1: 0.324
train loss:100%[**************************************************->]0.458
[epoch 97] train_loss: 0.532 train_accuracy: 0.823 val_accuracy: 0.837  recall: 0.716  f1: 0.725
train loss:100%[**************************************************->]1.2408
[epoch 98] train_loss: 0.729 train_accuracy: 0.798 val_accuracy: 0.497  recall: 0.314  f1: 0.279
train loss:100%[**************************************************->]0.345
[epoch 99] train_loss: 0.532 train_accuracy: 0.819 val_accuracy: 0.885  recall: 0.754  f1: 0.764
train loss:100%[**************************************************->]0.072
[epoch 100] train_loss: 0.420 train_accuracy: 0.856 val_accuracy: 0.865  recall: 0.731  f1: 0.744
Finished Training
